In [ ]:
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / 'src')) # 

from models.tbg_bm import BistritzMacDonaldTBG

from utils.kpath import make_k_path


 ## Moire BZ k-path



 Path: K -> Gamma -> M -> K

In [ ]:
# Magic angle
model = BistritzMacDonaldTBG(theta=1.05, n_shells=4, valley=+1)
hs = model.high_symmetry_points()
print(f"Nq = {model.Nq}, n_bands = {model.n_bands}")

path_labels = ['K', '\u0393', 'M', 'K']
path_pts = [hs[label] for label in path_labels]
kvec = make_k_path(path_pts, nkdensity=900)
Nk = len(kvec)

# Compute k-distances for x-axis ticks
dists = [0.0]
for i in range(1, len(path_pts)):
    d = np.linalg.norm(path_pts[i] - path_pts[i-1])
    dists.append(dists[-1] + d)
seg_dists = np.array(dists)

print(f"k-path: {Nk} points, total length = {dists[-1]:.3f} (1/A)")



In [ ]:
# Solve for all k-points
E_k = np.zeros((Nk, model.n_bands))
for i in range(Nk):
    Ei, _ = model.solve(kvec[i])
    E_k[i] = Ei

# Shift to CNP (charge neutrality point)
mid = model.n_bands // 2
mu_cnp = (np.max(E_k[:, mid-1]) + np.min(E_k[:, mid])) / 2
E_k_shifted = E_k - mu_cnp
print(f"mu_CNP = {mu_cnp:.4f} eV")
print(f"Bandwidth of central 4 bands: {np.max(E_k[:, mid-2:mid+2]) - np.min(E_k[:, mid-2:mid+2]):.4f} eV")



In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))

# Plot all bands
x = np.linspace(0, seg_dists[-1], Nk)
for b in range(model.n_bands):
    ax.plot(x, E_k_shifted[:, b], 'k-', lw=0.6, alpha=0.7)

ax.axhline(0, color='gray', ls='--', lw=0.8)
for d in seg_dists[1:-1]:
    ax.axvline(d, color='gray', ls=':', lw=0.6)

tick_labels = [r'K', r'$\Gamma$', r'M', r'K']
ax.set_xticks(seg_dists)
ax.set_xticklabels(tick_labels)
ax.set_xlim(0, seg_dists[-1])
ax.set_ylabel(r'$E - \mu_{\mathrm{CNP}}$ (eV)')
ax.set_title(rf'tBLG $\theta = 1.05^\circ$ (magic angle), $N_q = {model.Nq}$')
ax.set_ylim(-0.3, 0.3)

fig.tight_layout()
plt.show()



 ## Near-CNP zoom + valley comparison



 Are K and K' valley bands degenerate in the moire BZ?

In [ ]:
model_Kp = BistritzMacDonaldTBG(theta=1.05, n_shells=4, valley=-1)
E_k_Kp = np.zeros((Nk, model_Kp.n_bands))
for i in range(Nk):
    Ei, _ = model_Kp.solve(kvec[i])
    E_k_Kp[i] = Ei
mu_cnp_Kp = (np.max(E_k_Kp[:, mid-1]) + np.min(E_k_Kp[:, mid])) / 2
E_k_Kp_s = E_k_Kp - mu_cnp_Kp

# K-valley diff at K point (first point in path)
diff_K = np.max(np.abs(np.sort(E_k[0]) - np.sort(E_k_Kp[0])))
print(f"K vs K' diff at K-point: {diff_K:.2e} eV (should be ~0)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

n_show = 8
for b in range(mid - n_show//2, mid + n_show//2):
    ax1.plot(x, E_k_shifted[:, b], 'b-', lw=0.8, alpha=0.7)
    ax2.plot(x, E_k_Kp_s[:, b], 'r-', lw=0.8, alpha=0.7)

for ax in [ax1, ax2]:
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    for d in seg_dists[1:-1]:
        ax.axvline(d, color='gray', ls=':', lw=0.6)
    ax.set_xticks(seg_dists)
    ax.set_xticklabels(tick_labels)
    ax.set_xlim(0, seg_dists[-1])
    ax.set_ylim(-0.05, 0.05)

ax1.set_title(r'K valley ($\xi=+1$)')
ax1.set_ylabel(r'$E - \mu_{\mathrm{CNP}}$ (eV)')
ax2.set_title(r"K' valley ($\xi=-1$)")

fig.tight_layout()
plt.show()



 ## Twist angle comparison



 1.05 deg (magic) vs 2 deg vs 3 deg vs 5 deg -- flat band disappearance

In [ ]:
angles = [1.05, 2.0, 3.0, 5.0]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)

for ax, theta in zip(axes, angles):
    m = BistritzMacDonaldTBG(theta=theta, n_shells=4, valley=+1)
    hs_a = m.high_symmetry_points()
    path_a = make_k_path([hs_a[lbl] for lbl in path_labels], nkdensity=900)
    Ea = np.zeros((len(path_a), m.n_bands))
    for i in range(len(path_a)):
        Ei, _ = m.solve(path_a[i])
        Ea[i] = Ei
    
    mid_a = m.n_bands // 2
    mu_a = (np.max(Ea[:, mid_a-1]) + np.min(Ea[:, mid_a])) / 2
    Ea_s = Ea - mu_a
    
    xa = np.linspace(0, 1, len(path_a))
    for b in range(m.n_bands):
        ax.plot(xa, Ea_s[:, b], 'k-', lw=0.5, alpha=0.6)
    
    ax.axhline(0, color='gray', ls='--', lw=0.6)
    ax.set_title(rf'$\theta = {theta}^\circ$, $N_q={m.Nq}$')
    ax.set_xlabel('K -> Gamma -> M -> K')
    ax.set_ylim(-0.3, 0.3)

axes[0].set_ylabel(r'$E - \mu_{\mathrm{CNP}}$ (eV)')
fig.suptitle('tBLG band structure vs twist angle', y=1.02)
fig.tight_layout()
plt.show()



 ## Shell cutoff convergence



 Do the low-energy bands converge as n_shells increases?

In [ ]:
shells = [1, 2, 3, 4,]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)

for ax, ns in zip(axes, shells):
    try:
        m = BistritzMacDonaldTBG(theta=1.05, n_shells=ns, valley=+1)
    except Exception as e:
        ax.text(0.5, 0.5, f'n_shells={ns}\nfailed: {e}', 
                transform=ax.transAxes, ha='center', fontsize=10)
        continue
    hs_a = m.high_symmetry_points()
    path_a = make_k_path([hs_a[lbl] for lbl in path_labels], nkdensity=600)
    Ea = np.zeros((len(path_a), m.n_bands))
    for i in range(len(path_a)):
        Ei, _ = m.solve(path_a[i])
        Ea[i] = Ei
    
    mid_a = m.n_bands // 2
    mu_a = (np.max(Ea[:, mid_a-1]) + np.min(Ea[:, mid_a])) / 2
    Ea_s = Ea - mu_a
    
    xa = np.linspace(0, 1, len(path_a))
    for b in range(mid_a - 10, mid_a + 10):
        ax.plot(xa, Ea_s[:, b], 'k-', lw=0.5, alpha=0.6)
    
    ax.axhline(0, color='gray', ls='--', lw=0.6)
    ax.set_title(f'n_shells={ns}, Nq={m.Nq}')
    ax.set_xlabel('K -> Gamma -> M -> K')
    ax.set_ylim(-0.2, 0.2)

axes[0].set_ylabel(r'$E - \mu_{\mathrm{CNP}}$ (eV)')
fig.suptitle('BZ cutoff convergence at 1.05 deg', y=1.02)
fig.tight_layout()
plt.show()



 ## Expected results



 - **1.05 deg magic angle**: 2 flat bands near CNP, bandwidth <= ~0.04 eV

 - **K vs K' degeneracy**: diff ~ 0 (time-reversal symmetry)

 - **Large angle (5 deg)**: flat bands gone, recover Dirac-like dispersion

 - **n_shells >= 4**: low-energy bands converged; n_shells=3 cutoff insufficient

